In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
import pickle

In [2]:
data=pd.read_csv('Churn_Modelling.csv')
data.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [3]:
data=data.drop(['RowNumber', 'CustomerId', 'Surname'], axis=1)

In [5]:
label_encoder_gender=LabelEncoder()
data['Gender']=label_encoder_gender.fit_transform(data['Gender'])

In [11]:
onehot_encoder_geo=OneHotEncoder(handle_unknown='ignore')
geo_encoder=onehot_encoder_geo.fit_transform(data[['Geography']]).toarray()
geo_encoded_df=pd.DataFrame(geo_encoder, columns=onehot_encoder_geo.get_feature_names_out(['Geography']))

In [12]:
data=pd.concat([data.drop('Geography', axis=1), geo_encoded_df], axis=1)
data.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0


In [13]:
X=data.drop('EstimatedSalary', axis=1)
y=data['EstimatedSalary']

In [14]:
X_train, X_test, y_train, y_test=train_test_split(X, y, test_size=0.25, random_state=40)
scaler=StandardScaler()
X_train=scaler.fit_transform(X_train)
X_test=scaler.transform(X_test)

In [15]:
with open('label_encoder_gender.pkl', 'wb') as file:
    pickle.dump(label_encoder_gender, file)

with open('onehot_encoder_geo.pkl', 'wb') as file:
    pickle.dump(onehot_encoder_geo, file)

with open('scaler.pkl', 'wb') as file:
    pickle.dump(scaler, file)

In [16]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

I0000 00:00:1791615120.775066  557010 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1791615120.783408  557010 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1791615121.771049  557010 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1791615126.842730  557010 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.

In [18]:
model=Sequential([
    Dense(64, activation='relu', input_shape=(X_train.shape[1],)),
    Dense(32, activation='relu'),
    Dense(1)
])

In [21]:
model.compile(optimizer='adam', loss='mean_absolute_error', metrics=['mae'])
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_3 (Dense)                 │ (None, 64)             │           832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,945 (11.50 KB)

 Trainable params: 2,945 (11.50 KB)

 Non-trainable params: 0 (0.00 B)

In [22]:
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard
import datetime
log_dir='regressionlogs/fit'+datetime.datetime.now().strftime('%Y%m%d-%H%M%S')
#log_dir='/content/drive/MyDrive/logs/fit/'+datetime.datetime.now().strftime('%Y%m%d-%H%M%S')
tensorflow_callback=TensorBoard(log_dir=log_dir,histogram_freq=1)

In [23]:
early_stopping_callback=EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

In [24]:
history=model.fit(
    X_train, y_train, validation_data=(X_test, y_test),
    epochs=100,
    callbacks=[early_stopping_callback, tensorflow_callback]
)

Epoch 1/100
235/235 ━━━━━━━━━━━━━━━━━━━━ 6s 12ms/step - loss: 100192.0547 - mae: 100192.0547 - val_loss: 99490.9688 - val_mae: 99490.9688
Epoch 2/100
235/235 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - loss: 99592.9609 - mae: 99592.9609 - val_loss: 98261.3672 - val_mae: 98261.3672
Epoch 3/100
235/235 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - loss: 97458.0312 - mae: 97458.0312 - val_loss: 95061.3984 - val_mae: 95061.3984
Epoch 4/100
235/235 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - loss: 93148.3359 - mae: 93148.3359 - val_loss: 89554.2031 - val_mae: 89554.2031
Epoch 5/100
235/235 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - loss: 86617.1172 - mae: 86617.1172 - val_loss: 82132.7656 - val_mae: 82132.7656
Epoch 6/100
235/235 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - loss: 78606.4688 - mae: 78606.4688 - val_loss: 73847.5547 - val_mae: 73847.5547
Epoch 7/100
235/235 ━━━━━━━━━━━━━━━━━━━━ 3s 9ms/step - loss: 70205.5859 - mae: 70205.5859 - val_loss: 66089.5078 - val_mae: 66089.5078
Epoch 8/100
235/235 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/s

In [25]:
%load_ext tensorboard

In [29]:
%tensorboard --logdir regressionlogs/fit

Reusing TensorBoard on port 6006 (pid 614345), started 0:01:27 ago. (Use '!kill 614345' to kill it.)

In [31]:
test_loss, test_mae=model.evaluate(X_test, y_test)
print('Test MAE:',test_mae)

79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 50520.4141 - mae: 50520.4141
Test MAE: 50520.4140625


In [32]:
model.save('regresstion_model.keras')